In [37]:
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
import os

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

In [38]:
TARGET = "deterioration_next_12h"

# 🔥 FN-FOCUSED ENSEMBLE WEIGHTS
XGB_WEIGHT = 0.6
TCN_WEIGHT = 0.4

# 🔥 Threshold search range
THRESHOLDS = np.linspace(0.1, 0.5, 60)

In [39]:
df = pd.read_csv("../data/features/features.csv")

EXCLUDE = ["patient_id", TARGET]
FEATURES = [col for col in df.columns if col not in EXCLUDE]

In [40]:
import torch.nn as nn

class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        
        self.conv = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation
        )
        self.relu = nn.ReLU()
    
    def forward(self, x):
        out = self.conv(x)
        out = out[:, :, :-self.conv.padding[0]]
        return self.relu(out)


class TCN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        
        self.tcn = nn.Sequential(
            TCNBlock(num_features, 64, dilation=1),
            TCNBlock(64, 64, dilation=2),
            TCNBlock(64, 64, dilation=4),
            TCNBlock(64, 64, dilation=8),
        )
        
        self.fc = nn.Linear(64, 1)
    
    def forward(self, x):
        x = x.permute(0, 2, 1)
        out = self.tcn(x)
        out = out[:, :, -1]
        return self.fc(out).squeeze(-1)

In [41]:
class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, df, features, seq_len=6):
        self.X, self.y, self.indices = [], [], []
        
        for pid, group in df.groupby("patient_id"):
            group = group.sort_values("hour_from_admission")
            
            data = group[features].apply(pd.to_numeric, errors='coerce').values
            target = group[TARGET].values
            idxs = group.index.values
            
            for i in range(len(group) - seq_len):
                self.X.append(data[i:i+seq_len])
                self.y.append(target[i+seq_len-1])
                self.indices.append(idxs[i+seq_len-1])
        
        self.X = torch.tensor(np.nan_to_num(self.X), dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.indices[idx]

In [42]:
def get_fold_predictions(fold):

    val_df = pd.read_csv(f"../data/splits/val_fold_{fold}.csv")
    val_df = val_df.reset_index(drop=True)

    # -------- TCN --------
    tcn_model = TCN(len(FEATURES))
    tcn_model.load_state_dict(
        torch.load(f"../models/tcn/best_tcn_fold_{fold}.pt", map_location="cpu")
    )
    tcn_model.eval()

    val_ds = TimeSeriesDataset(val_df, FEATURES)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=64)

    tcn_preds_dict = {}

    with torch.no_grad():
        for X, _, idx in val_loader:
            preds = torch.sigmoid(tcn_model(X)).numpy()
            for i, p in zip(idx.numpy(), preds):
                tcn_preds_dict[i] = p

    # -------- XGBOOST --------
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model(f"../models/xgboost/xgb_fold_{fold}.json")

    X_val = val_df[FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
    xgb_preds = xgb_model.predict_proba(X_val)[:, 1]

    # -------- ALIGN --------
    aligned_tcn, aligned_xgb, aligned_y = [], [], []

    for i in range(len(val_df)):
        if i in tcn_preds_dict:
            aligned_tcn.append(tcn_preds_dict[i])
            aligned_xgb.append(xgb_preds[i])
            aligned_y.append(val_df[TARGET].iloc[i])

    return np.array(aligned_tcn), np.array(aligned_xgb), np.array(aligned_y)

In [43]:
all_tcn, all_xgb, all_y = [], [], []

for fold in range(5):
    print(f"Processing Fold {fold}")
    tcn_p, xgb_p, y = get_fold_predictions(fold)
    
    all_tcn.extend(tcn_p)
    all_xgb.extend(xgb_p)
    all_y.extend(y)

all_tcn = np.array(all_tcn)
all_xgb = np.array(all_xgb)
all_y = np.array(all_y)

Processing Fold 0
Processing Fold 1
Processing Fold 2
Processing Fold 3
Processing Fold 4


In [44]:
ensemble_preds = XGB_WEIGHT * all_xgb + TCN_WEIGHT * all_tcn

print("TCN PR-AUC:", average_precision_score(all_y, all_tcn))
print("XGB PR-AUC:", average_precision_score(all_y, all_xgb))
print("Ensemble PR-AUC:", average_precision_score(all_y, ensemble_preds))

TCN PR-AUC: 0.6381175028641722
XGB PR-AUC: 0.7143103651761331
Ensemble PR-AUC: 0.7097968556747255


In [45]:
best_t = 0
best_score = 0

for t in THRESHOLDS:
    preds_bin = (ensemble_preds >= t).astype(int)
    score = fbeta_score(all_y, preds_bin, beta=2)
    
    if score > best_score:
        best_score = score
        best_t = t

print("Best Threshold (F2):", best_t)
print("Best F2 Score:", best_score)

Best Threshold (F2): 0.4661016949152543
Best F2 Score: 0.7181105621232895


In [46]:
for t in np.linspace(0.1, 0.5, 20):
    preds = (ensemble_preds >= t).astype(int)
    
    r = recall_score(all_y, preds)
    p = precision_score(all_y, preds)
    fn = ((all_y == 1) & (preds == 0)).sum()
    fp = ((all_y == 0) & (preds == 1)).sum()
    
    print(f"{t:.2f} → Recall={r:.3f}, Precision={p:.3f}, FN={fn}, FP={fp}")

0.10 → Recall=0.981, Precision=0.142, FN=277, FP=83968
0.12 → Recall=0.969, Precision=0.171, FN=446, FP=66903
0.14 → Recall=0.954, Precision=0.198, FN=661, FP=54808
0.16 → Recall=0.939, Precision=0.224, FN=871, FP=46261
0.18 → Recall=0.925, Precision=0.248, FN=1067, FP=39790
0.21 → Recall=0.912, Precision=0.272, FN=1246, FP=34709
0.23 → Recall=0.900, Precision=0.296, FN=1428, FP=30483
0.25 → Recall=0.887, Precision=0.317, FN=1609, FP=27127
0.27 → Recall=0.876, Precision=0.340, FN=1768, FP=24202
0.29 → Recall=0.866, Precision=0.361, FN=1909, FP=21749
0.31 → Recall=0.855, Precision=0.383, FN=2068, FP=19541
0.33 → Recall=0.846, Precision=0.404, FN=2189, FP=17716
0.35 → Recall=0.837, Precision=0.427, FN=2319, FP=15991
0.37 → Recall=0.826, Precision=0.448, FN=2474, FP=14456
0.39 → Recall=0.815, Precision=0.470, FN=2631, FP=13048
0.42 → Recall=0.805, Precision=0.492, FN=2774, FP=11816
0.44 → Recall=0.795, Precision=0.515, FN=2909, FP=10665
0.46 → Recall=0.784, Precision=0.534, FN=3072, FP=97

In [90]:
final_preds = (ensemble_preds >= 0.24).astype(int)

tcn_support = (all_tcn >= 0.5)

final_preds = np.where(tcn_support, final_preds, final_preds)

precision = precision_score(all_y, final_preds)
recall = recall_score(all_y, final_preds)
f1 = f1_score(all_y, final_preds)

print("\nFinal Metrics:")
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


Final Metrics:
Precision: 0.30979482169027844
Recall: 0.8918500808663244
F1 Score: 0.459853882271895


In [91]:
cm = confusion_matrix(all_y, final_preds)

print("\nConfusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nBreakdown:")
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


Confusion Matrix:
[[208770  28257]
 [  1538  12683]]

Breakdown:
TN: 208770
FP: 28257
FN: 1538
TP: 12683


In [93]:
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../models", exist_ok=True)

np.save("../outputs/final_preds.npy", ensemble_preds)
np.save("../outputs/targets.npy", all_y)

import joblib
joblib.dump(0.24, "../models/best_threshold.pkl")

print("Saved predictions and threshold")

Saved predictions and threshold
